# Imports

In [42]:
import pandas as pd
import numpy as np

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

import os

import sqlite3
import faiss

# Load data

In [43]:
CSV_PATH = "../data/csv/processed/"
files    = os.listdir(CSV_PATH)
csvs     = sorted([file.split('_')[1].split('.')[0] for file in files])

historic_csvs = csvs[:-1]
train_df      = pd.concat([pd.read_csv(f"{CSV_PATH}products_{csv}.csv") for csv in historic_csvs], ignore_index=True)

latest_csv = csvs[-1]
test_df    = pd.read_csv(f"{CSV_PATH}products_{latest_csv}.csv")

In [44]:
df = pd.read_csv("../data/csv/mail_groceries.csv")

conn = sqlite3.connect("../data/sql/swipes.db")
labels = pd.read_sql_query("SELECT data_id, is_liked, is_superliked, is_passed FROM swipes", conn)
conn.close()

In [45]:
labels

,data_id,is_liked,is_superliked,is_passed
0,10876682,0,0,1
1,10808108,1,0,0
2,10863250,0,1,0
3,10888400,0,0,1
4,10808240,0,0,1
...,...,...,...,...
614,10888399,0,0,1
615,10820837,0,0,1
616,10821205,0,0,1
617,10820996,0,1,0


# Pre-processing 

In [46]:
vectorizer = CountVectorizer()

vectorizer = CountVectorizer(binary=True, lowercase=True)
train_sentences = train_df['product_name'].tolist()
train_product_encoding = vectorizer.fit_transform(train_sentences).toarray()

test_sentences = test_df['product_name'].tolist()
test_product_encoding = vectorizer.transform(test_sentences).toarray()

In [47]:
onehot = ['brand','category']
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown='ignore'), onehot),
    ("num", StandardScaler(), ['price'])
])
train_onehot = preprocessor.fit_transform(train_df).toarray()
test_onehot  = preprocessor.transform(test_df).toarray()
num_categories = sum(len(c) for c in preprocessor.named_transformers_['cat'].categories_)
print(f"Number of categories after one-hot encoding: {num_categories}")

Number of categories after one-hot encoding: 764


In [48]:
train_preprocessed = np.hstack((train_product_encoding, train_onehot))
test_preprocessed  = np.hstack((test_product_encoding, test_onehot))

In [49]:
train_df_preprocessed = pd.DataFrame(train_preprocessed, columns=[f"feature_{i}" for i in range(train_preprocessed.shape[1])])
train_df_preprocessed = pd.merge(train_df, train_df_preprocessed, left_index=True, right_index=True)

test_df_preprocessed = pd.DataFrame(test_preprocessed, columns=[f"feature_{i}" for i in range(test_preprocessed.shape[1])])
test_df_preprocessed = pd.merge(test_df, test_df_preprocessed, left_index=True, right_index=True)

In [50]:
train_df_features = train_df_preprocessed.iloc[:, len(train_df.columns):]
train_df_features['data_id'] = train_df_preprocessed['data_id']
train_df_features = pd.merge(train_df_features, labels, on='data_id', how='inner')

Xtrain = train_df_features.drop(columns=['data_id', 'is_liked', 'is_superliked', 'is_passed']).to_numpy()
ytrain = (train_df_features['is_liked'] | train_df_features['is_superliked']).to_numpy()

# ======================================================================0

test_df_features  = test_df_preprocessed.iloc[:, len(test_df.columns):]
test_df_features['data_id'] = test_df_preprocessed['data_id']

Xtest = test_df_features.drop(columns=['data_id']).to_numpy()

# Inference with KNN

In [51]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')

# Duplicate samples based on weights
weights = np.where(train_df_features['is_superliked'] == 1, 3, 1)

# Repeat each row according to its weight
indices = np.repeat(np.arange(len(train_df_features)), weights)

Xtrain_weighted = Xtrain[indices]
ytrain_weighted = ytrain[indices]
train_df_features_weighted = train_df_features.iloc[indices].reset_index(drop=True)

knn.fit(Xtrain_weighted, ytrain_weighted)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'cosine'
,metric_params,None
,n_jobs,None


In [52]:
train_df_features.shape

(652, 2001)

In [56]:
display(pd.merge(test_df_features[:13]['data_id'], test_df, on='data_id', how='left'))
knn.predict_proba(Xtest[:13])

,data_id,price,brand,category,product_name,units,quantity,unit_type,store_name,image_url,start_date,end_date,public_urls,translated_product,tinder_bio
0,10888291,16.00,DAVA Foods,Skalæg,Økologiske æg M/L,1,8,bk.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Organic eggs M/L,"✨ Eco-friendly omelet master, ready to hatch a..."
1,10888666,24.00,Energizer,Ladere & batterier,Batterier,1,10,pk.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Batteries,"✨⚡ Zap the chaos with a little power, and neve..."
2,10888298,6.00,NaN,Grønt,Kartofler,1,2,ps.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Potatoes,"I'm a humble potato, always ready to roast, bo..."
3,10888445,7.00,Exotic,Sodavand,Sodavand,1,1,fl.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Soda water,"Fizzing fun, never boring — just a little bubb..."
4,10888448,7.00,Squash,Sodavand,Appelsinsodavand Sukkerfri,1,1,fl.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Orange soda Sugar-free,"Sip on sunshine, sweet and free—orange soda su..."
5,10888443,7.00,Raspberry,Sodavand,Hindbærsodavand,1,1,fl.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Raspberry soda,"Sip, savor, and scream—raspberry soda is the f..."
6,10888447,7.00,Squash,Sodavand,Appelsinsodavand,1,1,fl.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Orange soda,"Citrusy, bubbly, and perfectly ripe—just like ..."
7,10888442,7.00,Coca Cola Zero,Sodavand,Cola,1,1,fl.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Cola,"Sip, savor, and never forget the thrill of a p..."
8,10888279,7.00,Coca Cola,Sodavand,Cola,1,1,fl.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Cola,"Fizzing fun, never boring — I'm the one that k..."
9,10888278,4.95,Arla,Mælk,Sødmælk,1,1,stk.,Netto,https://static.tilbudsugen.dk/1st-retail/2025/...,2025-12-20,2025-12-23,https://res.cloudinary.com/dfqzmnlga/image/upl...,Whole milk,"Smooth, creamy, and perfectly balanced — like ..."


array([[0.6, 0.4],
       [0.6, 0.4],
       [0. , 1. ],
       [0.8, 0.2],
       [0.6, 0.4],
       [0.6, 0.4],
       [0.6, 0.4],
       [0.8, 0.2],
       [0.8, 0.2],
       [0.4, 0.6],
       [0. , 1. ],
       [0.6, 0.4],
       [0. , 1. ]])

In [57]:
# get neighbors' product names according to knn
D, I = knn.kneighbors(Xtest[:13])
for i, neighbors in enumerate(I):
    print(pd.merge(train_df_features_weighted.iloc[neighbors]['data_id'], train_df, on='data_id', how='left')['product_name'].tolist())
    print(train_df_features_weighted.iloc[neighbors]['is_liked'].tolist())
    print(train_df_features_weighted.iloc[neighbors]['is_superliked'].tolist())
    print(D[i])
    print()


['Økologiske æg M/L', 'Energidrik', 'Laurberblade ', 'Affaldsposer', 'Chokolademousse']
[1, 0, 0, 1, 0]
[0, 0, 0, 0, 0]
[0.24603136 0.88284424 0.88415473 0.89076651 0.89087627]

['Energidrik', 'Laurberblade ', 'Affaldsposer', 'Bagepapir', 'Chokolademousse']
[0, 0, 1, 1, 0]
[0, 0, 0, 0, 0]
[0.89476797 0.89594509 0.90188394 0.90198253 0.90198253]

['Kartofler', 'Kartofler', 'Kartofler', 'Kartofler', 'Gulerødder']
[1, 0, 0, 0, 0]
[0, 1, 1, 1, 1]
[0.00116962 0.00116962 0.00116962 0.00116962 0.27546435]

['Sodavand', 'Sodavand', 'Cola', 'Cola', 'Cola']
[1, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0.00066474 0.00632927 0.5688786  0.5688786  0.5688786 ]

['Sodavand', 'Cola', 'Cola', 'Sportssodavand', 'Cola']
[1, 0, 0, 1, 0]
[0, 0, 0, 0, 0]
[0.61891395 0.61891395 0.61891395 0.61891395 0.61891395]

['Sodavand', 'Cola', 'Cola', 'Sportssodavand', 'Cola']
[1, 0, 0, 1, 0]
[0, 0, 0, 0, 0]
[0.5688786 0.5688786 0.5688786 0.5688786 0.5688786]

['Sodavand', 'Cola', 'Cola', 'Sportssodavand', 'Cola']
[1, 0, 0, 1, 0]


In [55]:
# Probabilities for all test samples, and print in descending order of probability of being liked
probas = knn.predict_proba(Xtest)
sorted_indices = np.argsort(-probas[:, 1])  # Sort by probability of being liked (class 1)
for idx in sorted_indices:
    data_id = test_df_features.iloc[idx]['data_id']
    product_name = test_df[test_df['data_id'] == data_id]['product_name'].values[0]
    print(f"{probas[idx, 1]:.4f} {product_name}")



1.0000 Kartofler
1.0000 Rødkål
1.0000 Clementiner
1.0000 Rødløg
1.0000 Krabbesalat
1.0000 Finthakket  Skinkesalat
1.0000 Kylling & Baconsalat
1.0000 Æggesalat
1.0000 Æggesalat 30% mindre fedt
1.0000 Flæskesteg
1.0000 Hønsesalat
1.0000 Laks i skiver
1.0000 Lighter
0.8000 Støvsugerposer
0.8000 Kartofler
0.8000 Torskerogn
0.8000 Flæskesteg
0.8000 Ribbensteg
0.8000 Tun i vand
0.8000 Tun i solsikkeolie
0.8000 Hakket grisekød 8-12%
0.6000 Rejer
0.6000 Røget Medister
0.6000 And
0.6000 Flæskesteg
0.6000 Citron Yoghurt
0.6000 Medister
0.6000 Appelsinjuice
0.6000 Vitamin Smoothie
0.6000 Appelsinjuice uden frugtkød
0.6000 Rejer
0.6000 Sødmælk
0.6000 Rullepølse
0.6000 Hamburgerryg
0.6000 Salami Snacks
0.6000 Gravlys
0.6000 Cashewnødder
0.6000 Valnødder
0.6000 Vanilje Yoghurt 
0.6000 Mandler
0.4000 Fokus Smoothie
0.4000 Appelsinsodavand
0.4000 Jordbær & Rababer Yoghurt 
0.4000 Skovbær Yoghurt 
0.4000 LED fyrfadslys
0.4000 Ice Tea Fersken
0.4000 Piskefløde 36%
0.4000 Appelsinsodavand Sukkerfri
0.400